# Biomni on RunPod

This notebook mirrors the Biomni launch pattern from the docs, but points the agent at a RunPod-hosted SGLang endpoint instead of `localhost`.

Use this notebook when you have already started SGLang inside a RunPod pod and exposed port `30000`. The public proxy URL usually looks like `https://<POD_ID>-30000.proxy.runpod.net/v1`.

In [ ]:
import os
from pprint import pprint

import requests

from biomni.agent import A1
from biomni.config import default_config

# Optional: install missing dependencies in the notebook environment.
# %pip install -U biomni requests runpod

# Database queries (indexes, retrieval, etc.) still use the default config.
default_config.llm = os.getenv('BIOMNI_DB_LLM', 'claude-3-5-sonnet-20241022')
default_config.source = os.getenv('BIOMNI_DB_SOURCE', 'Anthropic')

# Point Biomni reasoning to the RunPod-hosted SGLang OpenAI-compatible endpoint.
runpod_pod_id = os.getenv('RUNPOD_POD_ID', 'YOUR_RUNPOD_POD_ID')
runpod_port = os.getenv('RUNPOD_SGLANG_PORT', '30000')
runpod_base_url = os.getenv(
    'RUNPOD_SGLANG_BASE_URL',
    f'https://{runpod_pod_id}-{runpod_port}.proxy.runpod.net/v1',
)
runpod_api_key = os.getenv('RUNPOD_SGLANG_API_KEY', 'EMPTY')

print('Biomni database config:')
pprint({'llm': default_config.llm, 'source': default_config.source})
print()
print('RunPod SGLang endpoint:')
print(runpod_base_url)

In [ ]:
# Verify that the OpenAI-compatible endpoint is reachable before creating the agent.
models_url = runpod_base_url.rstrip('/') + '/models'
response = requests.get(models_url, timeout=30)
response.raise_for_status()
print('Endpoint check passed.')
pprint(response.json())

## Launch SGLang inside RunPod

Run this inside the RunPod pod shell to serve Biomni-R0 on port `30000`.

```bash
python -m sglang.launch_server --model-path RyanLi0802/Biomni-R0-Preview --port 30000 --host 0.0.0.0 --mem-fraction-static 0.8 --tp 2 --trust-remote-code --json-model-override-args '{"rope_scaling":{"rope_type":"yarn","factor":1.0,"original_max_position_embeddings":32768}, "max_position_embeddings": 131072}'
```

If you use a different pod ID, keep the notebook `RUNPOD_SGLANG_BASE_URL` value aligned with the RunPod proxy URL.

In [ ]:
# Biomni reasoning uses the RunPod endpoint; database queries still use default_config above.
agent = A1(
    path='./data',
    llm='biomni/Biomni-R0-32B-Preview',
    source='Custom',
    base_url=runpod_base_url,
    api_key=runpod_api_key,
)

prompt = 'Plan a CRISPR screen to identify genes regulating T cell exhaustion'
log, answer = agent.go(prompt)

print('Prompt:')
print(prompt)
print()
print('Answer:')
print(answer)